# Step 0 : Queries for RAGAS Evaluation

In [1]:
import os
from dotenv import load_dotenv

# Load FIRST, before any other imports
load_dotenv("/Users/midhunln/Documents/rag20march_with_eval/Ingestion_plus_Retriever_eval/ingestion.env")

# Verify it's loaded
print("API Key loaded:", bool(os.getenv("OPENAI_API_KEY")))


API Key loaded: True


In [2]:
questions = [
    "What is the purpose of the SEBI Master Circular for compliance with the LODR Regulations, 2015 by listed entities?",
    "Which earlier master circular is superseded by the November 11, 2024 LODR master circular?",
    "What are the three categories into which holding of specified securities must be divided under the LODR master circular?",
    "How is total shareholding calculated for the purpose of computing public shareholding under the LODR master circular?",
    "How is percentage of promoter shareholding calculated under the LODR master circular?"
]

In [3]:
import os
import sys

# Get the absolute path to the project root directory
# Assuming the notebook is at rag_pipeline/ragas_eval/dataset.ipynb
# We need to go up two levels to reach the project root.
project_root = os.path.abspath(os.path.join(os.getcwd(), '../../')) # go two levels up

# Add the project root to sys.path
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Now your imports should work
from rag_pipeline.workflow.config import Settings
from rag_pipeline.workflow.service import RAGService
# ... other imports

# Step 1 : Instantiate the workflow

In [4]:
from rag_pipeline.workflow.config import Settings
from rag_pipeline.workflow.service import RAGService
from rag_pipeline.workflow.graph import RAGWorkflow
from rag_pipeline.workflow.node_orchestrator import Nodes
from rag_pipeline.workflow.database.sessions import Database
from rag_pipeline.workflow.repositories.pinecone_repository import PineconeRepository
from rag_pipeline.workflow.embeddings.sentence_transformer_embedding import (
    SentenceTransformerEmbedding,
)
from rag_pipeline.workflow.embeddings.sparse_embedding import (
    SentenceTransformerSparseEmbedding,
)
from rag_pipeline.workflow.llms.ollama_llama import OllamaLLM
from rag_pipeline.workflow.database.db_repositories.conversation_repository import (
    ConversationRepository,
)
from rag_pipeline.workflow.configs.pinecone_config import PineconeConfig
from rag_pipeline.workflow.configs.llm_config import LLMConfig
from rag_pipeline.api.routes import ask_endpoint
from rag_pipeline.workflow.embeddings.openai_embedding import OpenAIEmbedding
from rag_pipeline.workflow.llms.openai import OpenAILLM
from rag_pipeline.workflow.llms.finetuned_llm import FinetunedLLM

from dotenv import load_dotenv
load_dotenv("/Users/midhunln/Documents/rag20march_with_eval/Ingestion_plus_Retriever_eval/ingestion.env")
import logging

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
)
logger = logging.getLogger(__name__)

settings = Settings()


# Database
database = Database(settings.database_url)
logger.info("Database initialized")

# Embeddings configuration
pinecone_config = PineconeConfig()

# Embedding strategies
"""
dense_embedding = SentenceTransformerEmbedding(pinecone_config)
"""
dense_embedding = OpenAIEmbedding()
sparse_embedding = SentenceTransformerSparseEmbedding(pinecone_config)
logger.info("Embedding strategies initialized")

# Vector Database
vector_db = PineconeRepository(
    api_key=os.getenv("PINECONE_API_KEY"),
    pinecone_config=pinecone_config,
    dense_embedding_strategy=dense_embedding,
    sparse_embedding_strategy=sparse_embedding,
    environment=settings.pinecone_environment,
)

logger.info("Vector database initialized")

# LLM
llm_config = LLMConfig(model_name=settings.finetuned_model_path)
"""
llm = OpenAILLM(llm_config)
"""
llm = FinetunedLLM(llm_config)

logger.info("LLM initialized")

# Repository
conversation_repo = ConversationRepository()
logger.info("Conversation repository initialized")

# Service
service = RAGService(
    database=database,
    vector_db=vector_db,
    conversation_repository=conversation_repo,
    llm=llm
)
logger.info("RAG service initialized")

# Workflow
nodes = Nodes(service=service)
workflow = RAGWorkflow(nodes=nodes)

/Users/midhunln/Documents/rag20march_with_eval/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-04-09 16:26:22,030 - __main__ - INFO - Database initialized
2026-04-09 16:26:22,112 - sentence_transformers.SentenceTransformer - INFO - Use pytorch device_name: mps
2026-04-09 16:26:22,112 - sentence_transformers.SentenceTransformer - INFO - Load pretrained SparseEncoder: naver/splade-cocondenser-ensembledistil
2026-04-09 16:26:22,641 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/naver/splade-cocondenser-ensembledistil/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-04-09 16:26:22,847 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/naver/splade-cocondenser-ensembledistil/49cf4c7b0db5b870a401ddf5e2669993ef3699c7/modules.json "HTTP/1.1

# Step 2 : Create the raw form of raga dataset

In [5]:
"""
dataset = [
    {
        "user_input": "What is the refund policy?",
        "retrieved_contexts": [
            "Refunds are allowed within 30 days of purchase.",
            "Refunds are processed to the original payment method."
        ],
        "response": "You can get a refund within 30 days, and it will be sent to your original payment method."
    },
    {
        "user_input": "Can I upgrade my plan?",
        "retrieved_contexts": [
            "Users can upgrade their subscription at any time.",
        ],
        "response": "Yes, you can upgrade anytime."
    }
]
"""

'\ndataset = [\n    {\n        "user_input": "What is the refund policy?",\n        "retrieved_contexts": [\n            "Refunds are allowed within 30 days of purchase.",\n            "Refunds are processed to the original payment method."\n        ],\n        "response": "You can get a refund within 30 days, and it will be sent to your original payment method."\n    },\n    {\n        "user_input": "Can I upgrade my plan?",\n        "retrieved_contexts": [\n            "Users can upgrade their subscription at any time.",\n        ],\n        "response": "Yes, you can upgrade anytime."\n    }\n]\n'

In [6]:
def raga_dataset_creation(workflow, questions):
    ragas_raw_dataset = []
    for question in questions:
        response = workflow.execute(question, session_id = 1)

        dict1= {
            "user_inputs" : response["query"],
            "retrieved_contexts" : [doc.page_content for doc in response["retrieved_documents"]],
            "response" : response["response"]
        }
        ragas_raw_dataset.append(dict1)
    
    return ragas_raw_dataset

ragas_raw_dataset = raga_dataset_creation(workflow, questions)


Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
2026-04-09 16:26:37,942 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
Batches: 100%|██████████| 1/1 [00:00<00:00,  5.35it/s]
2026-04-09 16:26:40,017 - rag_pipeline.workflow.service - INFO - Retrieved 10 documents for query
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/te

In [7]:
ragas_raw_dataset[0]

{'user_inputs': 'What is the purpose of the SEBI Master Circular for compliance with the LODR Regulations, 2015 by listed entities?',
 'retrieved_contexts': ['Page 1 of 28 \n \n \nMASTER CIRCULAR \n \n \nSEBI/HO/ISD/ISD-PoD-2/P/CIR/2024/126                   September 23, 2024\n  \n                          \nTo  \n \n1. All Recognized Stock Exchanges \n \n2. All Depositories \n \n3. All Listed Companies \n \n4. All Market Intermediaries (MIs) registered with SEBI under Section 12 of the SEBI Act, \n1992 \n \n5. Fiduciaries as per Securities and Exchange Board of India (Prohibition of Insider \nTrading) Regulations, 2015  \n \nDear Sir/Madam, \n \nSub: Master Circular on Surveillance of Securities Market \n \n1. Securities and Exchange Board of India (SEBI) has been issuing various circulars from time \nto time pertaining to effective surveillance of the securities market. A Master Circular in the \nform of a compilation of provisions of all the relevant circulars, was last issued on t

# Step 3 : Convert to RAGAS Objects 

In [8]:
from ragas import EvaluationDataset, SingleTurnSample

samples = []

for row in ragas_raw_dataset:
    samples.append(
        SingleTurnSample(
            user_input=row["user_inputs"],
            response=row["response"],
            retrieved_contexts=row["retrieved_contexts"]
        )
    )

eval_dataset = EvaluationDataset(samples=samples)

# Step 4 : Start Ragas Evaluation with local llm and embeddings

In [9]:
import os

from ragas import EvaluationDataset, SingleTurnSample, evaluate
from ragas.metrics import Faithfulness, AnswerRelevancy

from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# Ensure API key exists
assert os.getenv("OPENAI_API_KEY")

# ---- Create default LLM & embeddings ----
llm = LangchainLLMWrapper(
    ChatOpenAI(model="gpt-4o-mini")   # default-ish lightweight model
)

embeddings = LangchainEmbeddingsWrapper(
    OpenAIEmbeddings()
)


/var/folders/lz/6vnq1xf93h912vrdhnbwvvmc0000gn/T/ipykernel_55979/662960238.py:4: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import Faithfulness, AnswerRelevancy
/var/folders/lz/6vnq1xf93h912vrdhnbwvvmc0000gn/T/ipykernel_55979/662960238.py:4: DeprecationWarning: Importing AnswerRelevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import AnswerRelevancy
  from ragas.metrics import Faithfulness, AnswerRelevancy
/var/folders/lz/6vnq1xf93h912vrdhnbwvvmc0000gn/T/ipykernel_55979/662960238.py:15: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gp

In [10]:
result = evaluate(
    dataset=eval_dataset,
    metrics=[
        Faithfulness(llm=llm),
        AnswerRelevancy(llm=llm, embeddings=embeddings),
    ],
    show_progress=True,
)

print(result)


Evaluating:   0%|          | 0/10 [00:00<?, ?it/s]2026-04-09 16:40:47,939 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-09 16:40:49,879 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-04-09 16:40:50,959 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
Evaluating:  10%|█         | 1/10 [00:05<00:51,  5.76s/it]2026-04-09 16:40:51,844 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-09 16:40:51,846 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-09 16:40:51,847 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-09 16:40:51,848 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-09 16:40:51,849 - httpx - INFO - HTTP Request: PO

{'faithfulness': 0.2333, 'answer_relevancy': 0.9561}


In [11]:
print(result)

{'faithfulness': 0.2333, 'answer_relevancy': 0.9561}
